In [3]:
import os
import numpy as np
import random
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.optimizers import Adam
import tensorflow as tf

In [4]:
def load_fingerprint(path, target_size=(160, 160)):
    img = load_img(path, color_mode='grayscale', target_size=target_size)
    img = img_to_array(img) / 255.0
    return img

In [5]:
def create_pairs(image_dir):
    data = {}
    for file in os.listdir(image_dir):
        if file.endswith(".bmp"):
            person_id = file.split('_')[0]
            if person_id not in data:
                data[person_id] = []
            data[person_id].append(os.path.join(image_dir, file))
    
    pairs = []
    labels = []

    person_ids = list(data.keys())

    for pid in person_ids:
        images = data[pid]
        # Положительные пары
        for i in range(len(images) - 1):
            img1 = load_fingerprint(images[i])
            img2 = load_fingerprint(images[i + 1])
            pairs.append([img1, img2])
            labels.append(1)

        # Отрицательные пары
        for _ in range(len(images) - 1):
            other_pid = random.choice([p for p in person_ids if p != pid])
            img1 = load_fingerprint(random.choice(images))
            img2 = load_fingerprint(random.choice(data[other_pid]))
            pairs.append([img1, img2])
            labels.append(0)

    return np.array(pairs), np.array(labels)

In [6]:
def build_base_network(input_shape):
    input = Input(shape=input_shape)
    x = layers.Conv2D(64, (7, 7), activation='relu')(input)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(128, (5, 5), activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Flatten()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dense(128)(x)
    return Model(input, x)

def euclidean_distance(vects):
    x, y = vects
    return tf.sqrt(tf.reduce_sum(tf.square(x - y), axis=1, keepdims=True))

def contrastive_loss(y_true, y_pred):
    margin = 1
    return tf.reduce_mean(y_true * tf.square(y_pred) + (1 - y_true) * tf.square(tf.maximum(margin - y_pred, 0)))

In [7]:
pairs, labels = create_pairs("dataset/train_data")

# Разделяем
img1 = pairs[:, 0]
img2 = pairs[:, 1]

# Сиамская модель
input_shape = (160, 160, 1)
base_network = build_base_network(input_shape)

input_a = Input(shape=input_shape)
input_b = Input(shape=input_shape)

feat_a = base_network(input_a)
feat_b = base_network(input_b)

distance = layers.Lambda(euclidean_distance)([feat_a, feat_b])
model = Model(inputs=[input_a, input_b], outputs=distance)

model.compile(loss=contrastive_loss, optimizer=Adam(0.0001))
model.summary()

# Обучение
model.fit([img1, img2], labels, batch_size=32, epochs=10)

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)    │ (None, 160, 160, 1)       │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ input_layer_2 (InputLayer)    │ (None, 160, 160, 1)       │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ functional (Functional)       │ (None, 128)               │      42,708,608 │ input_layer_1[0][0],       │
│                               │                           │                 │ input_layer_2[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ lambda (Lambda)               │ (None, 1)                 │               0 │ functional[0][0],          │
│                               │                           │                 │ functional[1][0]           │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 42,708,608 (162.92 MB)

 Trainable params: 42,708,608 (162.92 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 190s 4s/step - loss: 0.2482
Epoch 2/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 133s 3s/step - loss: 0.1366
Epoch 3/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 132s 3s/step - loss: 0.1032
Epoch 4/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 129s 3s/step - loss: 0.0721
Epoch 5/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 126s 3s/step - loss: 0.0654
Epoch 6/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 120s 2s/step - loss: 0.0516
Epoch 7/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 122s 2s/step - loss: 0.0452
Epoch 8/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 122s 2s/step - loss: 0.0409
Epoch 9/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 121s 2s/step - loss: 0.0435
Epoch 10/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 121s 2s/step - loss: 0.0336


In [8]:
from sklearn.metrics import accuracy_score
preds = model.predict([img1, img2])
pred_labels = (preds < 0.5).astype("int").flatten()
true_labels = labels.flatten()

accuracy = accuracy_score(true_labels, pred_labels)
print(f"✅ Accuracy on training pairs: {accuracy:.4f}")

50/50 ━━━━━━━━━━━━━━━━━━━━ 24s 478ms/step
✅ Accuracy on training pairs: 0.9981


In [30]:
def compare_fingerprints(img_path1, img_path2,threshold=0.3):
    img1 = np.expand_dims(load_fingerprint(img_path1), axis=0)
    img2 = np.expand_dims(load_fingerprint(img_path2), axis=0)
    distance = model.predict([img1, img2])[0][0]
    print(f"🔎 Distance: {distance:.4f}")
    if distance < threshold:
        print("✅ SAME person")
    else:
        print("❌ DIFFERENT persons")

In [38]:
compare_fingerprints(
    "dataset/real_data/00006.bmp",
    "dataset/real_data/00008.bmp"
)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 105ms/step
🔎 Distance: 0.9791
❌ DIFFERENT persons
